In [ ]:
# import necessary libraries needed for fine-tuning
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import pandas as pd
from datasets import Dataset
import evaluate
import numpy as np

# load the preprocessed local CSVs
train_df = pd.read_csv("../data/cleaned/bbc/train.csv")
test_df = pd.read_csv("../data/cleaned/bbc/test.csv")

train_df_clean = train_df.dropna(subset=["Content", "Summary"]) 
test_df_clean = test_df.dropna(subset=["Content", "Summary"])

# convert pandas dataframe into a hugging face dataset object
train_dataset = Dataset.from_pandas(train_df_clean.reset_index(drop=True)) # deleting the old index column entirely
test_dataset = Dataset.from_pandas(test_df_clean.reset_index(drop=True)) # deleting the old index column entirely

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("../models/bart-base", use_fast=True) # "use_fast" makes use of the faster tokenizerw 

# define tokenization function
def tokenize_function(dataset):
    # turns the input text into numerical tokens
    inputs = tokenizer(
        dataset["Content"],
        max_length=512,
        truncation=True,
        padding=False
    )

    # tokenize the labels (summary)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            dataset["Summary"],
            max_length=128,
            truncation=True,
            padding=False
        )

    inputs["labels"] = labels["input_ids"]
    return inputs

# tokenizing the hugging face dataset object
tokenized_train = train_dataset.map(tokenize_function, batched=True) # runs in a batch of 1000 by default
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Load the pre-trained BART model
model = AutoModelForSeq2SeqLM.from_pretrained("../models/bart-base/")
model.gradient_checkpointing_enable() # saves GPU VRAM by recomputing some activations on the fly during back propagation, to compute gradients.

# data collator
data_collation = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model) # helps in dynamic padding of rows in a single batch

# define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="../outputs/fine-tuned-bart/", # output directory for the train outputs
    evaluation_strategy="epoch", # tells the trainer when to run the evaluation strat
    save_strategy="epoch", # save the version after each epoch
    save_total_limit=2, # saves two versions
    load_best_model_at_end=True,
    metric_for_best_model="rougeL", 
    greater_is_better=True,

    learning_rate=3e-5, # controls how big of a step the optimizer takes when updating the model weights 
    per_device_train_batch_size=1, # number of samples processed in parallel per GPU during training
    per_device_eval_batch_size=1, # batch size for validation forward passes
    gradient_accumulation_steps=8, # Accumulate gradients for 8 mini-batches before updating weights -> simulates a larger effective batch size without extra GPU memory
    gradient_checkpointing=True, # Save GPU memory by recomputing certain activations during backpropagation instead of storing them
    fp16=True, # Uses mixed precision training (16-bit floats instead of 32-bit floats) if GPU supports it (NVIDIA GPU with Tensor Cores). Speeds up training.

    num_train_epochs=3, # number of full passes through the dataset
    weight_decay=0.01, # Discourages weights from growing too large to prevent overfitting
    predict_with_generate=True, # during evaluation, it makes the model generate summaries (instead of just computing loss)
    logging_dir="../outputs/fine-tuned-bart/logs/",
    logging_strategy="epoch" 
)

# load the ROUGE score metric
rouge = evaluate.load("rouge")
# bert = evaluate.load("bertscore")

# define compute_metrics() function
def compute_metrics(eval_pred):
    # generate predictions
    # 1. obtain the predictions and labels
    predictions, labels = eval_pred

    # 2. Replace -100 with pad_token_id for decoding tokens into text
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 3. decode to tokens into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 4. ROUGE requires newline 
    decoded_preds = ["\n".join(preds.strip().split(". ")) for preds in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split(". ")) for label in decoded_labels]

    # 5. compute ROUGE/BERTScore
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels,use_stemmer=True, use_aggregator=True)
    # bertscore_result = bert.compute(predictions=decoded_preds, references=decoded_labels, lang="en")

    # 6. create a single dictionary of metrics
    result = {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        # "bertscore_f1": np.mean(bertscore_result["f1"]),
    }

    # 7. return in percent
    return {k: round(v * 100, 4) for k, v in result.items()}

# for smoke test purposes
tiny_train = tokenized_train.select(range(64))
tiny_eval  = tokenized_test.select(range(32))

# config the trainer
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tiny_train,    
    eval_dataset = tiny_eval,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics,
    data_collator=data_collation,
)

# run the trainer
trainer.train()

# evaluate the trainer
eval_metrics = trainer.evaluate()
print("eval: ", eval_metrics)

# save the fine-tuned model + tokenizer
model.save_pretrained("../outputs/fine-tuned-bart")
tokenizer.save_pretrained("../outputs/fine-tuned-bart")


Could not locate the tokenizer configuration file, will try to use the model config instead.
loading configuration file ../models/bart-base\config.json
Model config BartConfig {
  "_name_or_path": "../models/bart-base",
  "activation_dropout": 0.1,
  "activation_function": "gelu",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "BartModel"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "classif_dropout": 0.1,
  "classifier_dropout": 0.0,
  "d_model": 768,
  "decoder_attention_heads": 12,
  "decoder_ffn_dim": 3072,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": true,
  "encoder_attention_heads": 12,
  "encoder_ffn_dim": 3072,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 2,
  "forced_bos_token_id": 0,
  "forced_eos_token_id": 2,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "i

{'loss': 2.3265, 'learning_rate': 2.25e-05, 'epoch': 1.0}



                                               



                                  




 33%|███▎      | 8/24 [02:32<02:34,  9.69s/it] 






Saving model checkpoint to ../outputs/fine-tuned-bart/checkpoint-8
Configuration saved in ../outputs/fine-tuned-bart/checkpoint-8\config.json


{'eval_loss': 1.4115557670593262, 'eval_rouge1': 22.2338, 'eval_rouge2': 11.8971, 'eval_rougeL': 18.9987, 'eval_runtime': 75.5657, 'eval_samples_per_second': 0.423, 'eval_steps_per_second': 0.423, 'epoch': 1.0}


Model weights saved in ../outputs/fine-tuned-bart/checkpoint-8\pytorch_model.bin
tokenizer config file saved in ../outputs/fine-tuned-bart/checkpoint-8\tokenizer_config.json
Special tokens file saved in ../outputs/fine-tuned-bart/checkpoint-8\special_tokens_map.json
                                               



                                  




 67%|██████▋   | 16/24 [04:01<01:40, 12.59s/it]






The following columns in the evaluation set don't have a corresponding argument in `BartForConditionalGeneration.forward` and have been ignored: Summary, Content. If Summary, Content are not expected by `BartForConditionalGeneration.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 32
  Batch size = 1


...

{'train_runtime': 483.8257, 'train_samples_per_second': 0.397, 'train_steps_per_second': 0.05, 'train_loss': 1.8947024742762248, 'epoch': 3.0}


100%|██████████| 32/32 [01:13<00:00,  2.29s/it]
Configuration saved in ../outputs/fine-tuned-bart\config.json


eval:  {'eval_loss': 1.4115557670593262, 'eval_rouge1': 22.2338, 'eval_rouge2': 11.8971, 'eval_rougeL': 18.9987, 'eval_runtime': 76.211, 'eval_samples_per_second': 0.42, 'eval_steps_per_second': 0.42, 'epoch': 3.0}


Model weights saved in ../outputs/fine-tuned-bart\pytorch_model.bin
tokenizer config file saved in ../outputs/fine-tuned-bart\tokenizer_config.json
Special tokens file saved in ../outputs/fine-tuned-bart\special_tokens_map.json


('../outputs/fine-tuned-bart\\tokenizer_config.json',
 '../outputs/fine-tuned-bart\\special_tokens_map.json',
 '../outputs/fine-tuned-bart\\vocab.json',
 '../outputs/fine-tuned-bart\\merges.txt',
 '../outputs/fine-tuned-bart\\added_tokens.json',
 '../outputs/fine-tuned-bart\\tokenizer.json')